# 01 — De notícias a features

**Objetivo:** transformar notícias não estruturadas (`1 linha = 1 notícia`) em sinais numéricos diários (`1 linha = 1 dia`) que os modelos conseguem utilizar.

### Entrada

`data/silver/news/br_energy_news.csv`

### Saída

`data/gold/features/br_energy_news_features_daily.csv`

### Features iniciais

- `news_count`
- `signal`
- `avg_score`
- `avg_relevance`
- `positive_count`
- `neutral_count`
- `negative_count`
- `positive_share`
- `neutral_share`
- `negative_share`
- `max_relevance`
- `score_std`
- `signal_ma_3`
- `signal_ma_5`
- `signal_ma_7`

O `signal` diário segue a lógica:

```text
Σ(score × relevance)
────────────────────
    Σ(relevance)
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NEWS_PATH = PROJECT_ROOT / "data/silver/news/br_energy_news.csv"
SCORES_PATH = PROJECT_ROOT / "data/silver/nlp/br_energy_news_scores.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/gold/features/br_energy_news_features_daily.csv"

print("News input :", NEWS_PATH)
print("Scores input:", SCORES_PATH)
print("Output     :", OUTPUT_PATH)


## 1. Carregar as notícias


In [ ]:
news = pd.read_csv(NEWS_PATH)
scores = pd.read_csv(SCORES_PATH)
scores['_row_order'] = range(len(scores))
scores['_analyzed_at'] = pd.to_datetime(
    scores['analyzed_at'], utc=True, errors='coerce'
)
scores = (
    scores.sort_values(
        ['article_id', '_analyzed_at', '_row_order'],
        na_position='first',
    )
    .drop_duplicates('article_id', keep='last')
    .drop(columns=['_row_order', '_analyzed_at'])
)
news = news.merge(scores, on='article_id', how='inner', validate='one_to_one')

news["published_at"] = pd.to_datetime(
    news["published_at"],
    utc=True,
    errors="coerce",
)

news["score"] = pd.to_numeric(
    news["score"],
    errors="coerce",
)

news["relevance"] = pd.to_numeric(
    news["relevance"],
    errors="coerce",
)

print(f"Rows: {len(news):,}")
print(f"Period: {news['published_at'].min()} -> {news['published_at'].max()}")

news.head()


## 2. Filtrar notícias analisadas

Para gerar features de sentimento, consideramos somente notícias com `score` e `relevance` disponíveis.

In [ ]:
analyzed = news.dropna(
    subset=["published_at", "score", "relevance"]
).copy()

analyzed["date"] = analyzed["published_at"].dt.date

print(f"Analyzed news: {len(analyzed):,}")
print(f"Unique days  : {analyzed['date'].nunique():,}")

analyzed[
    [
        "date",
        "source",
        "title",
        "sentiment",
        "score",
        "relevance",
    ]
].head()


## 3. Contribuição ponderada por notícia

Cada notícia contribui para o sinal diário por:

`weighted_score = score × relevance`

A relevância funciona como peso na agregação.

In [ ]:
analyzed["weighted_score"] = (
    analyzed["score"]
    * analyzed["relevance"]
)

analyzed[
    [
        "date",
        "title",
        "score",
        "relevance",
        "weighted_score",
    ]
].head()


## 4. Features diárias

Agregamos as notícias por `published_at`.

A granularidade final passa a ser diária


In [ ]:
sentiment_daily = (
    analyzed
    .assign(
        sentiment=analyzed["sentiment"]
        .astype(str)
        .str.lower()
    )
    .groupby(
        ["date", "sentiment"]
    )
    .size()
    .unstack(fill_value=0)
)

for column in [
    "positive",
    "neutral",
    "negative",
]:
    if column not in sentiment_daily.columns:
        sentiment_daily[column] = 0

sentiment_daily = sentiment_daily[
    [
        "positive",
        "neutral",
        "negative",
    ]
].rename(
    columns={
        "positive": "positive_count",
        "neutral": "neutral_count",
        "negative": "negative_count",
    }
)

sentiment_daily.head()


In [ ]:
daily = (
    analyzed
    .groupby("date")
    .agg(
        news_count=("score", "count"),
        weighted_sum=("weighted_score", "sum"),
        total_relevance=("relevance", "sum"),
        avg_score=("score", "mean"),
        avg_relevance=("relevance", "mean"),
        max_relevance=("relevance", "max"),
        score_std=("score", "std"),
    )
    .join(sentiment_daily)
    .reset_index()
)

daily["signal"] = np.where(
    daily["total_relevance"] > 0,
    daily["weighted_sum"] / daily["total_relevance"],
    0.0,
)

daily["positive_share"] = (
    daily["positive_count"] / daily["news_count"]
)

daily["neutral_share"] = (
    daily["neutral_count"] / daily["news_count"]
)

daily["negative_share"] = (
    daily["negative_count"] / daily["news_count"]
)

daily["score_std"] = daily["score_std"].fillna(0.0)

daily.head()


## 5. Médias móveis

As médias móveis ajudam a reduzir ruído diário e capturar persistência do sinal.

Começamos com janelas simples de 3, 5 e 7 dias de observação.


In [ ]:
daily = daily.sort_values("date").reset_index(drop=True)

for window in [3, 5, 7]:
    daily[f"signal_ma_{window}"] = (
        daily["signal"]
        .rolling(
            window=window,
            min_periods=1,
        )
        .mean()
    )

daily[
    [
        "date",
        "signal",
        "signal_ma_3",
        "signal_ma_5",
        "signal_ma_7",
    ]
].tail(10)


## 6. Dataset final

In [ ]:
feature_columns = [
    "date",
    "news_count",
    "signal",
    "avg_score",
    "avg_relevance",
    "max_relevance",
    "score_std",
    "positive_count",
    "neutral_count",
    "negative_count",
    "positive_share",
    "neutral_share",
    "negative_share",
    "signal_ma_3",
    "signal_ma_5",
    "signal_ma_7",
]

daily_features = daily[feature_columns].copy()

numeric_columns = daily_features.select_dtypes(
    include="number"
).columns

daily_features[numeric_columns] = (
    daily_features[numeric_columns]
    .round(4)
)

daily_features.tail(10)


## 7. Validações rápidas

- uma linha por dia;
- `signal` entre -1 e +1;
- shares entre 0 e 1;
- contagem de sentimentos igual a `news_count`.


In [ ]:
assert daily_features["date"].is_unique

assert daily_features["signal"].between(
    -1,
    1,
).all()

for column in [
    "positive_share",
    "neutral_share",
    "negative_share",
]:
    assert daily_features[column].between(
        0,
        1,
    ).all()

sentiment_total = (
    daily_features["positive_count"]
    + daily_features["neutral_count"]
    + daily_features["negative_count"]
)

assert (
    sentiment_total
    == daily_features["news_count"]
).all()

print("✓ Validations passed")


## 8. Persistir features diárias

`02_market_news_eda.ipynb`


In [ ]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

daily_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(
    f"✓ Saved {len(daily_features):,} daily rows "
    f"to {OUTPUT_PATH}"
)


## 9. Resumo

O dataset final contém uma linha por dia com:

- intensidade e direção agregada das notícias;
- relevância média;
- volume de notícias;
- distribuição de sentimento;
- dispersão dos scores;
- médias móveis do sinal.


In [ ]:
daily_features.head()